## Quiz 11

### Imports

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

In [3]:
num = 1000 
 
event_time = int(num / 2) 
 
R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
 
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
 
results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
 
alpha, beta = results.params 
 
resid = R_target - results.predict(sm.add_constant(R_market)) 
 
print(resid[event_time + 1] / resid[:event_time].std(ddof = 2)) 

2.5412774041448807


### Q1

In [4]:
np.random.seed(42)

num = 1000
event_time = int(num / 2)
n_sims = 5000

alpha_level = 0.05
df = event_time - 2
t_crit = stats.t.ppf(1 - alpha_level / 2, df)

detections = 0

for _ in range(n_sims):
    event_time = int(num / 2) 
    
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
    
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market)) 

    t_stat = resid[event_time + 1] / resid[:event_time].std(ddof=2)

    if abs(t_stat) > t_crit:
        detections += 1

power = detections / n_sims
print(f"{power:.1f}")

0.5


### Q2

In [7]:
num = 1000
true_event_time = int(num / 2)
alpha_level = 0.05

detections = 0
total = 0

for placebo_time in range(10, num - 1):
    np.random.seed(0)

    R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
        
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market)) 

    df = placebo_time - 2
    if df < 1:
        continue

    t_crit = stats.t.ppf(1 - alpha_level / 2, df)

    t_stat = resid[placebo_time + 1] / resid[:placebo_time].std(ddof=2)

    total += 1
    if abs(t_stat) > t_crit:
        detections += 1

fraction_detected = detections / total
print(f"{detections / total:.2f}")

0.05


### Q3

In [10]:
num = 1000
true_event_time = int(num / 2)
n_placebo_each_side = 20
n_runs = 2000

fractions = []

for run in range(n_runs):
    np.random.seed(run)

    R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
            
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market))

    actual_t = resid[true_event_time + 1] / resid[:true_event_time].std(ddof=2)

    placebo_times = list(range(true_event_time - n_placebo_each_side, true_event_time)) + list(range(true_event_time + 1, true_event_time + 1 + n_placebo_each_side))

    placebo_t_stats = []
    for placebo_time in placebo_times:
        X_train_p = sm.add_constant(R_market[:placebo_time])
        results_p = sm.OLS(R_target[:placebo_time], X_train_p).fit()
        resid_p = R_target - results_p.predict(sm.add_constant(R_market))

        t_stat_p = resid_p[placebo_time + 1] / resid_p[:placebo_time].std(ddof=2)
        placebo_t_stats.append(t_stat_p)

    placebo_t_stats = np.array(placebo_t_stats)

    frac_higher = np.mean(np.abs(placebo_t_stats) > np.abs(actual_t))
    fractions.append(frac_higher)

fractions = np.array(fractions)
print(f"{fractions.mean():.2f}")

0.14


### Q4

In [14]:
def make_error(corr_const, num): 
 
 
  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
 
 
  err = list() 
 
 
  prev = np.random.normal(0, sigma) 
 
 
  for n in range(num): 
 
 
    prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
 
 
    err.append(prev) 
 
 
  return np.array(err) 

num = 1000
true_event_time = int(num / 2)
alpha_level = 0.05
corr_const = 0.9

detections = 0
total = 0

for placebo_time in range(10, num - 1):
    np.random.seed(0)

    R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
                
    R_target = 2 + R_market + make_error(corr_const, num) + (np.arange(num) == int(num / 2) + 1) * 2 
    
    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 
    
    alpha, beta = results.params 
    
    resid = R_target - results.predict(sm.add_constant(R_market))

    df = placebo_time - 2
    if df < 1:
        continue

    t_crit = stats.t.ppf(1 - alpha_level / 2, df)

    t_stat = resid[placebo_time + 1] / resid[:placebo_time].std(ddof=2)

    total += 1
    if abs(t_stat) > t_crit:
        detections += 1

print(f"{detections / total:.2f}")

0.03
